In [1]:
from ogb.nodeproppred.dataset_pyg import PygNodePropPredDataset

dataset = PygNodePropPredDataset(name="ogbn-arxiv")
data = dataset[0]

print(data)

c:\Users\MuhammedSaidali\anaconda3\Lib\site-packages\outdated\__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Data(num_nodes=169343, edge_index=[2, 1166243], x=[169343, 128], node_year=[169343, 1], y=[169343, 1])


In [2]:
print(data.x.shape)
print(data.edge_index.shape)
print(data.y.shape)

print(data.num_nodes)
print(data.num_edges)

print(data.x[:5])

print(data.y[:5])

print(data.node_year[:5])

torch.Size([169343, 128])
torch.Size([2, 1166243])
torch.Size([169343, 1])
169343
1166243
tensor([[-5.7943e-02, -5.2530e-02, -7.2603e-02, -2.6555e-02,  1.3044e-01,
         -2.4139e-01, -4.4924e-01, -1.8443e-02, -8.7218e-02,  1.1232e-01,
         -9.2125e-02, -2.8956e-01, -8.1012e-02,  7.4489e-02, -1.5620e-01,
         -9.7413e-02,  1.1937e-01,  6.4575e-01,  7.7375e-02, -9.3860e-02,
         -4.0037e-01,  3.1137e-01, -5.4176e-01,  8.0455e-02, -6.9500e-03,
          5.4232e-01, -1.2230e-02, -1.8077e-01,  1.6466e-02,  5.0778e-02,
         -2.0828e-01, -8.7010e-02,  1.2363e-02,  2.8167e-01,  1.0045e-01,
         -1.6425e-01,  2.6892e-02,  7.8199e-02,  7.9534e-02, -1.3387e-02,
          2.9149e-01,  4.1601e-02, -1.4137e-01, -1.3446e-01,  1.6178e-02,
          2.8096e-01, -9.1925e-02, -2.4031e-01,  4.6179e-01,  1.8732e-01,
          1.5335e-01,  3.3118e-02,  1.0760e-02,  1.2446e-02, -1.5886e-01,
          9.7980e-02,  3.0520e-02,  1.6234e-02, -9.5681e-02,  5.2140e-02,
          3.2184e-01, 

In [3]:
split_idx = dataset.get_idx_split()

train_idx = split_idx["train"]
valid_idx = split_idx["valid"]
test_idx = split_idx["test"]

print(train_idx.shape)
print(valid_idx.shape)
print(test_idx.shape)

torch.Size([90941])
torch.Size([29799])
torch.Size([48603])


In [5]:
print(data.y.unique())
print(len(data.y.unique()))
print(data.y.max())
print(data.num_features)
print(data.is_directed())
print(data.has_isolated_nodes())
print(data.has_self_loops())

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39])
40
tensor(39)
128
True
False
False


In [6]:
import torch
import torch.nn.functional as F

from torch_geometric.nn import GCNConv
from ogb.nodeproppred import Evaluator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [7]:
data = dataset[0].to(device)

split_idx = dataset.get_idx_split()

train_idx = split_idx["train"].to(device)
valid_idx = split_idx["valid"].to(device)
test_idx = split_idx["test"].to(device)

In [8]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):

        # First GCN layer
        x = self.conv1(x, edge_index)

        # Activation
        x = F.relu(x)

        # Prevent overfitting
        x = F.dropout(x, p=0.5, training=self.training)

        # Second GCN layer
        x = self.conv2(x, edge_index)

        return x

In [9]:
model = GCN(
    in_channels=data.num_features,
    hidden_channels=256,
    out_channels=40
).to(device)

print(model)

GCN(
  (conv1): GCNConv(128, 256)
  (conv2): GCNConv(256, 40)
)


In [10]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

In [11]:
criterion = torch.nn.CrossEntropyLoss()

def train():

    model.train()

    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = criterion(
        out[train_idx],
        data.y[train_idx].squeeze()
    )

    loss.backward()

    optimizer.step()

    return loss.item()

In [12]:
evaluator = Evaluator(name="ogbn-arxiv")

@torch.no_grad()
def test():

    model.eval()

    out = model(data.x, data.edge_index)

    pred = out.argmax(dim=-1, keepdim=True)

    train_acc = evaluator.eval({
        "y_true": data.y[train_idx],
        "y_pred": pred[train_idx],
    })["acc"]

    valid_acc = evaluator.eval({
        "y_true": data.y[valid_idx],
        "y_pred": pred[valid_idx],
    })["acc"]

    test_acc = evaluator.eval({
        "y_true": data.y[test_idx],
        "y_pred": pred[test_idx],
    })["acc"]

    return train_acc, valid_acc, test_acc

In [13]:
for epoch in range(1, 101):

    loss = train()

    train_acc, valid_acc, test_acc = test()

    if epoch % 10 == 0:
        print(
            f"Epoch: {epoch:03d} | "
            f"Loss: {loss:.4f} | "
            f"Train: {train_acc:.4f} | "
            f"Valid: {valid_acc:.4f} | "
            f"Test: {test_acc:.4f}"
        )

Epoch: 010 | Loss: 2.6161 | Train: 0.3601 | Valid: 0.4212 | Test: 0.4278
Epoch: 020 | Loss: 2.1212 | Train: 0.4689 | Valid: 0.4814 | Test: 0.4311
Epoch: 030 | Loss: 1.8549 | Train: 0.5275 | Valid: 0.5221 | Test: 0.4656
Epoch: 040 | Loss: 1.7034 | Train: 0.5545 | Valid: 0.5418 | Test: 0.4813
Epoch: 050 | Loss: 1.6246 | Train: 0.5699 | Valid: 0.5550 | Test: 0.4908
Epoch: 060 | Loss: 1.5770 | Train: 0.5811 | Valid: 0.5630 | Test: 0.4974
Epoch: 070 | Loss: 1.5468 | Train: 0.5880 | Valid: 0.5686 | Test: 0.5014
Epoch: 080 | Loss: 1.5265 | Train: 0.5927 | Valid: 0.5750 | Test: 0.5080
Epoch: 090 | Loss: 1.5124 | Train: 0.5972 | Valid: 0.5756 | Test: 0.5083
Epoch: 100 | Loss: 1.4953 | Train: 0.6003 | Valid: 0.5794 | Test: 0.5152
